# Clean faces for SMIRK (full run, LIMIT = 0)

Inputs: expw-crops, affectnet-crops. Settings: GPU T4 x1, Internet on. Run with Save Version -> Save & Run All, then Output -> New Dataset (clean-faces).

In [ ]:
import glob, os, sys, subprocess
print(sys.version)

DATASETS = {}
for meta in glob.glob('/kaggle/input/**/meta.csv', recursive=True):
    folder = os.path.dirname(meta)
    name = {'expw_crops': 'expw', 'affectnet_crops': 'affectnet'}.get(os.path.basename(folder))
    if name and os.path.isdir(os.path.join(folder, 'images')):
        DATASETS[name] = folder
print(DATASETS)
assert DATASETS, 'add expw-crops and/or affectnet-crops as input'

# quick test run on the first LIMIT images of every dataset; set LIMIT = 0 for the full run
LIMIT = 0

OUT = '/kaggle/working/clean_faces'
os.makedirs(OUT, exist_ok=True)

In [ ]:
def works(python):
    return subprocess.run([python, '-c', 'import mediapipe, cv2, tqdm'], capture_output=True).returncode == 0

MP_PYTHON = sys.executable
if not works(MP_PYTHON):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe'])
if not works(MP_PYTHON):
    print('MediaPipe does not work with this Python, creating a Python 3.11 environment')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '-q', '-p', '3.11', '/tmp/mpenv'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '-q', '-p', '/tmp/mpenv/bin/python',
                    'mediapipe==0.10.14', 'opencv-python-headless', 'tqdm', 'numpy<2'], check=True)
    MP_PYTHON = '/tmp/mpenv/bin/python'
assert works(MP_PYTHON), 'MediaPipe could not be installed'
print('MediaPipe runs with', MP_PYTHON)

!wget -q -nc https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task -O /kaggle/working/face_landmarker.task
!ls -la /kaggle/working/face_landmarker.task

In [ ]:
%%writefile /kaggle/working/extract_mediapipe_landmarks.py
"""
Run MediaPipe FaceLandmarker on a folder of face crops (e.g. the output of crop_expw.py / prepare_affectnet.py)
and store all landmarks in a single file, in the order of meta.csv.

The crops may contain neighbouring faces (they are enlarged around the labeled face), so up to --max_faces faces are
detected and the one closest to the image center is kept.

Output (.npz):
    names       (N,)        image names from meta.csv
    landmarks   (N, 478, 3) x, y in pixels and z as returned by MediaPipe (as in utils/mediapipe_utils.py); NaN if not detected
    detected    (N,)        whether a face was found
    num_faces   (N,)        number of detected faces
"""
import os
import argparse
import csv
import numpy as np
import cv2
from multiprocessing import Pool
from tqdm import tqdm

detector = None


def init_detector(model_path, max_faces):
    global detector
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision

    options = vision.FaceLandmarkerOptions(base_options=python.BaseOptions(model_asset_path=model_path),
                                           num_faces=max_faces)
    detector = vision.FaceLandmarker.create_from_options(options)


def process(image_path):
    import mediapipe as mp

    image = cv2.imread(image_path)
    if image is None:
        return None, 0
    h, w = image.shape[:2]

    result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(image, cv2.COLOR_BGR2RGB)))
    if not result.face_landmarks:
        return None, 0

    faces = [np.array([[l.x * w, l.y * h, l.z] for l in face], dtype=np.float32) for face in result.face_landmarks]
    center = np.array([w / 2, h / 2])
    best = min(faces, key=lambda f: np.linalg.norm(f[:, :2].mean(axis=0) - center))
    return best, len(faces)


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Extract MediaPipe landmarks for a folder of face crops.')
    parser.add_argument('--image_dir', type=str, required=True, help='Folder with the images listed in meta.csv')
    parser.add_argument('--meta', type=str, required=True, help='meta.csv with a "name" column (image file name without .png)')
    parser.add_argument('--out', type=str, required=True, help='Output .npz file')
    parser.add_argument('--model', type=str, default='assets/face_landmarker.task', help='MediaPipe face_landmarker.task')
    parser.add_argument('--max_faces', type=int, default=3)
    parser.add_argument('--num_processes', type=int, default=4)
    parser.add_argument('--limit', type=int, default=0, help='Only process the first N images (quick test run); 0 = all')
    args = parser.parse_args()

    with open(args.meta) as f:
        names = [row['name'] for row in csv.DictReader(f)]
    if args.limit:
        names = names[:args.limit]
    paths = [os.path.join(args.image_dir, name + '.png') for name in names]

    landmarks = np.full((len(names), 478, 3), np.nan, dtype=np.float32)
    num_faces = np.zeros(len(names), dtype=np.int32)

    with Pool(args.num_processes, initializer=init_detector, initargs=(args.model, args.max_faces)) as pool:
        for i, (lmk, n) in enumerate(tqdm(pool.imap(process, paths, chunksize=16), total=len(paths))):
            num_faces[i] = n
            if lmk is not None:
                landmarks[i] = lmk

    detected = num_faces > 0
    os.makedirs(os.path.dirname(os.path.abspath(args.out)), exist_ok=True)
    np.savez_compressed(args.out, names=np.array(names), landmarks=landmarks, detected=detected, num_faces=num_faces)
    print(f'Detected a face in {detected.sum()}/{len(names)} images ({100 * detected.mean():.1f}%), '
          f'{(num_faces > 1).sum()} images with several faces')

In [ ]:
%%writefile /kaggle/working/compute_face_quality.py
"""
Measure the quality of face crops, to filter them before SMIRK training. Needs the landmarks of
extract_mediapipe_landmarks.py.

For every image:
    black_fraction    fraction of pure black (0,0,0) pixels in the 224x224 window that SMIRK crops around the
                      MediaPipe landmarks (BaseDataset.crop_face with --scale); black pixels come from padding
    sharpness         variance of the Laplacian of that window (low = blurry)
    face_size         size of the MediaPipe landmark box in pixels

and, with face parsing (BiSeNet-ResNet18 trained on CelebAMask-HQ, https://github.com/yakhyo/face-parsing) of that
window, inside the convex hull of the 105 MediaPipe landmarks that SMIRK uses (brows, eyes, nose, mouth):
    occluded_fraction pixels that are not skin / brows / eyes / nose / mouth / lips (hands, objects, glasses, hair, ...)
    glasses_fraction  pixels labeled as eyeglasses / sunglasses
    hat_fraction      pixels labeled as hat
Parsing metrics are NaN when parsing is disabled.

Output: CSV with one row per image of meta.csv.
"""
import os
import csv
import argparse
import numpy as np
import cv2
from tqdm import tqdm

# indices of the MediaPipe landmarks that SMIRK uses (same as mediapipe_indices in datasets/base_dataset.py)
SMIRK_MEDIAPIPE_INDICES = [276, 282, 283, 285, 293, 295, 296, 300, 334, 336, 46, 52, 53,
        55, 63, 65, 66, 70, 105, 107, 249, 263, 362, 373, 374, 380,
       381, 382, 384, 385, 386, 387, 388, 390, 398, 466, 7, 33, 133,
       144, 145, 153, 154, 155, 157, 158, 159, 160, 161, 163, 173, 246,
       168, 6, 197, 195, 5, 4, 129, 98, 97, 2, 326, 327, 358,
         0, 13, 14, 17, 37, 39, 40, 61, 78, 80, 81, 82, 84,
        87, 88, 91, 95, 146, 178, 181, 185, 191, 267, 269, 270, 291,
       308, 310, 311, 312, 314, 317, 318, 321, 324, 375, 402, 405, 409,
       415]

def smirk_crop_matrix(landmarks, scale, image_size=224):
    # same crop as BaseDataset.crop_face / demo.py
    left, top = landmarks.min(axis=0)
    right, bottom = landmarks.max(axis=0)
    old_size = (right - left + bottom - top) / 2
    center = np.array([right - (right - left) / 2.0, bottom - (bottom - top) / 2.0])
    size = int(old_size * scale)
    src = np.array([[center[0] - size / 2, center[1] - size / 2], [center[0] - size / 2, center[1] + size / 2],
                    [center[0] + size / 2, center[1] - size / 2]], dtype=np.float32)
    dst = np.array([[0, 0], [0, image_size - 1], [image_size - 1, 0]], dtype=np.float32)
    return cv2.getAffineTransform(src, dst)


def image_metrics(image, landmarks, scale):
    M = smirk_crop_matrix(landmarks[:, :2], scale)
    window = cv2.warpAffine(image, M, (224, 224), flags=cv2.INTER_NEAREST, borderValue=0)
    black_fraction = float((window == 0).all(axis=2).mean())

    gray = cv2.cvtColor(cv2.warpAffine(image, M, (224, 224), flags=cv2.INTER_LINEAR, borderValue=0), cv2.COLOR_BGR2GRAY)
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    extent = landmarks[:, :2].max(axis=0) - landmarks[:, :2].min(axis=0)
    return black_fraction, sharpness, float(extent.max())


class FaceParser:
    """BiSeNet-ResNet18 face parsing (CelebAMask-HQ classes) from https://github.com/yakhyo/face-parsing (MIT)."""
    LABELS = ['background', 'skin', 'l_brow', 'r_brow', 'l_eye', 'r_eye', 'eye_g', 'l_ear', 'r_ear', 'ear_r',
              'nose', 'mouth', 'u_lip', 'l_lip', 'neck', 'neck_l', 'cloth', 'hair', 'hat']
    FACE_PARTS = ['skin', 'l_brow', 'r_brow', 'l_eye', 'r_eye', 'nose', 'mouth', 'u_lip', 'l_lip']
    SIZE = 512  # BiSeNet input size

    def __init__(self, device, repo_dir, weights):
        import sys
        import torch
        sys.path.insert(0, repo_dir)
        from models.bisenet import BiSeNet

        self.torch, self.device = torch, device
        self.net = BiSeNet(len(self.LABELS), backbone_name='resnet18')
        self.net.load_state_dict(torch.load(weights, map_location=device))
        self.net.to(device).eval()
        self.mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
        self.face_ids = [self.LABELS.index(n) for n in self.FACE_PARTS]
        self.glasses_id, self.hat_id = self.LABELS.index('eye_g'), self.LABELS.index('hat')

    def __call__(self, images, landmarks, scale):
        """images: list of HxWx3 BGR uint8; landmarks: list of 478x3 arrays. Returns a list of 3-tuples."""
        torch = self.torch
        # parse the window that SMIRK crops (the face fills it, as in CelebAMask-HQ)
        matrices = [smirk_crop_matrix(lmk[:, :2], scale, self.SIZE) for lmk in landmarks]
        windows = np.stack([cv2.warpAffine(im, M, (self.SIZE, self.SIZE), flags=cv2.INTER_LINEAR, borderValue=0)
                            for im, M in zip(images, matrices)])
        x = torch.from_numpy(windows[..., ::-1].copy()).to(self.device).permute(0, 3, 1, 2).float() / 255
        x = (x - self.mean) / self.std
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16, enabled=self.device != 'cpu'):
            labels = self.net(x)[0].argmax(dim=1).cpu().numpy()

        results = []
        for label_map, lmk, M in zip(labels, landmarks, matrices):
            points = lmk[SMIRK_MEDIAPIPE_INDICES, :2] @ M[:, :2].T + M[:, 2]
            region = np.zeros(label_map.shape, dtype=np.uint8)
            cv2.fillConvexPoly(region, cv2.convexHull(points.astype(np.int32)), 1)
            inside = label_map[region.astype(bool)]
            if inside.size == 0:
                results.append((np.nan, np.nan, np.nan))
                continue
            results.append((float(1 - np.isin(inside, self.face_ids).mean()),
                            float((inside == self.glasses_id).mean()),
                            float((inside == self.hat_id).mean())))
        return results


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Compute quality / occlusion metrics of face crops.')
    parser.add_argument('--image_dir', type=str, required=True)
    parser.add_argument('--landmarks', type=str, required=True, help='.npz from extract_mediapipe_landmarks.py')
    parser.add_argument('--out', type=str, required=True, help='Output CSV')
    parser.add_argument('--scale', type=float, default=1.6, help='SMIRK crop scale (test scale of the configs)')
    parser.add_argument('--no_parsing', action='store_true', help='Skip face parsing (no occlusion metrics)')
    parser.add_argument('--device', type=str, default='cuda')
    parser.add_argument('--batch_size', type=int, default=32)
    parser.add_argument('--parser_repo', type=str, default='/tmp/face_parsing', help='Clone of github.com/yakhyo/face-parsing')
    parser.add_argument('--parser_weights', type=str, default='/tmp/bisenet_resnet18.pt', help='resnet18.pt of its releases')
    args = parser.parse_args()

    data = np.load(args.landmarks)
    names, landmarks, detected = data['names'], data['landmarks'], data['detected']
    face_parser = None if args.no_parsing else FaceParser(args.device, args.parser_repo, args.parser_weights)

    columns = ['name', 'detected', 'face_size', 'black_fraction', 'sharpness', 'occluded_fraction', 'glasses_fraction', 'hat_fraction']
    rows = []
    batch = []

    def flush():
        parsed = face_parser([b[1] for b in batch], [b[2] for b in batch], args.scale) if face_parser else [(np.nan,) * 3] * len(batch)
        for (row, _, _), p in zip(batch, parsed):
            rows.append(row + list(p))
        batch.clear()

    for name, lmk, ok in tqdm(zip(names, landmarks, detected), total=len(names)):
        if not ok:
            rows.append([name, False] + [np.nan] * 6)
            continue
        image = cv2.imread(os.path.join(args.image_dir, f'{name}.png'))
        black_fraction, sharpness, face_size = image_metrics(image, lmk, args.scale)
        batch.append(([name, True, face_size, black_fraction, sharpness], image, lmk))
        if len(batch) == args.batch_size:
            flush()
    if batch:
        flush()

    # keep the order of the landmarks file
    order = {n: i for i, n in enumerate(names)}
    rows.sort(key=lambda r: order[r[0]])
    with open(args.out, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        writer.writerows(rows)
    print(f'Wrote {len(rows)} rows to {args.out}')

In [ ]:
%%writefile /kaggle/working/filter_and_split.py
"""
Keep the clean faces of one or more datasets and split them into train / val / test.

Each dataset is given as  name=meta.csv,landmarks.npz,quality.csv  (outputs of crop_expw.py / prepare_affectnet.py,
extract_mediapipe_landmarks.py and compute_face_quality.py). A face is kept when MediaPipe found it and its quality
metrics pass the thresholds (faces cut by the image border, i.e. with MediaPipe landmarks outside the image, are
rejected as well); NaN parsing metrics (parser found no face) count as failing unless --keep_unparsed.

Splits: datasets whose meta.csv already has train/val/test splits keep them (e.g. AffectNet); the others are split
per expression label with --val_fraction / --test_fraction (fixed seed).

Output (out_dir):
    clean_meta.csv     dataset, name, label, expression, split + quality metrics
    landmarks.npz      keys ("dataset/name") and MediaPipe landmarks (478x3) of the kept faces
    rejected.csv       dataset, name, reason   (first failing check)
"""
import os
import csv
import argparse
import numpy as np


def read_csv(path):
    with open(path) as f:
        return list(csv.DictReader(f))


def to_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Filter face crops by quality and split them.')
    parser.add_argument('--dataset', action='append', required=True, help='name=meta.csv,landmarks.npz,quality.csv')
    parser.add_argument('--out_dir', type=str, required=True)
    parser.add_argument('--min_face_size', type=float, default=80, help='MediaPipe landmark box size in pixels')
    parser.add_argument('--max_black_fraction', type=float, default=0.8)
    parser.add_argument('--min_sharpness', type=float, default=30)
    parser.add_argument('--max_occluded_fraction', type=float, default=0.12)
    parser.add_argument('--max_glasses_fraction', type=float, default=0.01)
    parser.add_argument('--max_hat_fraction', type=float, default=0.02)
    parser.add_argument('--max_landmarks_outside', type=float, default=0.05,
                        help='Reject faces cut by the image border: fraction of MediaPipe landmarks outside the image')
    parser.add_argument('--image_size', type=int, default=256, help='Size of the (square) face crops')
    parser.add_argument('--keep_unparsed', action='store_true', help='Keep faces without parsing metrics')
    parser.add_argument('--val_fraction', type=float, default=0.1)
    parser.add_argument('--test_fraction', type=float, default=0.1)
    parser.add_argument('--seed', type=int, default=0)
    args = parser.parse_args()

    # (column, failing test, reason)
    checks = [
        ('face_size', lambda v: v < args.min_face_size, 'face_too_small'),
        ('black_fraction', lambda v: v > args.max_black_fraction, 'black_border'),
        ('sharpness', lambda v: v < args.min_sharpness, 'blurry'),
        ('glasses_fraction', lambda v: v > args.max_glasses_fraction, 'glasses'),
        ('hat_fraction', lambda v: v > args.max_hat_fraction, 'hat'),
        ('occluded_fraction', lambda v: v > args.max_occluded_fraction, 'occluded'),
    ]
    parsing_columns = {'occluded_fraction', 'glasses_fraction', 'hat_fraction'}

    def landmarks_outside(lmk):
        xy = lmk[:, :2]
        return float(((xy < 0) | (xy >= args.image_size)).any(axis=1).mean())

    os.makedirs(args.out_dir, exist_ok=True)
    rng = np.random.default_rng(args.seed)
    clean_rows, rejected_rows, keys, kept_landmarks = [], [], [], []

    for spec in args.dataset:
        dataset, paths = spec.split('=', 1)
        meta_path, landmarks_path, quality_path = paths.split(',')
        meta = {r['name']: r for r in read_csv(meta_path)}
        quality = {r['name']: r for r in read_csv(quality_path)}
        lmk_data = np.load(landmarks_path)
        lmk_index = {n: i for i, n in enumerate(lmk_data['names'])}

        kept = []
        for name, m in meta.items():
            q = quality.get(name)
            reason = None
            if q is None:
                reason = 'not_processed'  # not in the quality file (e.g. landmarks computed with --limit)
            elif q['detected'] != 'True':
                reason = 'no_face'
            elif landmarks_outside(lmk_data['landmarks'][lmk_index[name]]) > args.max_landmarks_outside:
                reason = 'face_cut'
            else:
                for column, fails, why in checks:
                    value = to_float(q[column])
                    if np.isnan(value):
                        if column in parsing_columns and not args.keep_unparsed:
                            reason = 'not_parsed'
                            break
                        continue
                    if fails(value):
                        reason = why
                        break
            if reason:
                rejected_rows.append([dataset, name, reason])
            else:
                kept.append((name, m, q))

        # split: keep existing train/val/test, otherwise stratified random split per label
        has_split = all(m.get('split') in ('train', 'val', 'test') for _, m, _ in kept) and kept
        splits = {}
        if has_split:
            splits = {name: m['split'] for name, m, _ in kept}
        else:
            for label in sorted({m['label'] for _, m, _ in kept}):
                names = [name for name, m, _ in kept if m['label'] == label]
                rng.shuffle(names)
                n_test, n_val = int(round(len(names) * args.test_fraction)), int(round(len(names) * args.val_fraction))
                for i, name in enumerate(names):
                    splits[name] = 'test' if i < n_test else 'val' if i < n_test + n_val else 'train'

        for name, m, q in kept:
            clean_rows.append([dataset, name, m['label'], m['expression'], splits[name]] +
                              [q[c] for c, _, _ in checks])
            keys.append(f'{dataset}/{name}')
            kept_landmarks.append(lmk_data['landmarks'][lmk_index[name]])

        n_rejected = sum(r[0] == dataset for r in rejected_rows)
        print(f'{dataset}: kept {len(kept)} / {len(meta)} ({100 * len(kept) / max(len(meta), 1):.1f}%), '
              f'splits kept from meta.csv: {bool(has_split)}')
        reasons = {}
        for r in rejected_rows:
            if r[0] == dataset:
                reasons[r[2]] = reasons.get(r[2], 0) + 1
        print('  rejected:', dict(sorted(reasons.items(), key=lambda x: -x[1])), f'total {n_rejected}')

    with open(os.path.join(args.out_dir, 'clean_meta.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['dataset', 'name', 'label', 'expression', 'split'] + [c for c, _, _ in checks])
        writer.writerows(clean_rows)
    with open(os.path.join(args.out_dir, 'rejected.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['dataset', 'name', 'reason'])
        writer.writerows(rejected_rows)
    np.savez_compressed(os.path.join(args.out_dir, 'landmarks.npz'), keys=np.array(keys),
                        landmarks=np.array(kept_landmarks, dtype=np.float32).reshape(-1, 478, 3))
    print(f'Saved {len(clean_rows)} clean faces to {args.out_dir}')

In [ ]:
for name, folder in DATASETS.items():
    print('=====', name)
    !{MP_PYTHON} /kaggle/working/extract_mediapipe_landmarks.py --image_dir {folder}/images --meta {folder}/meta.csv --out {OUT}/{name}_landmarks.npz --model /kaggle/working/face_landmarker.task --num_processes 4 --limit {LIMIT}

In [ ]:
# face parsing model: BiSeNet-ResNet18 trained on CelebAMask-HQ (https://github.com/yakhyo/face-parsing, MIT license)
!git clone -q --depth 1 https://github.com/yakhyo/face-parsing.git /tmp/face_parsing 2>/dev/null || echo "already cloned"
!wget -q -nc https://github.com/yakhyo/face-parsing/releases/download/weights/resnet18.pt -O /tmp/bisenet_resnet18.pt
!ls -la /tmp/bisenet_resnet18.pt

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

for name, folder in DATASETS.items():
    print('=====', name)
    !{sys.executable} /kaggle/working/compute_face_quality.py --image_dir {folder}/images --landmarks {OUT}/{name}_landmarks.npz --out {OUT}/{name}_quality.csv --device {DEVICE} --parser_repo /tmp/face_parsing --parser_weights /tmp/bisenet_resnet18.pt

In [ ]:
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt

quality = pd.concat([pd.read_csv(f'{OUT}/{n}_quality.csv', dtype={'name': str}).assign(dataset=n) for n in DATASETS], ignore_index=True)
print(quality.groupby('dataset')['detected'].mean().rename('detected fraction'))
print(quality.describe(percentiles=[.05, .25, .5, .75, .95]).T)

def show(metric, values, n=8):
    q = quality[quality['detected']].dropna(subset=[metric])
    fig, axes = plt.subplots(len(values), n, figsize=(2 * n, 2.2 * len(values)))
    for row, v in zip(axes, values):
        pick = q.iloc[(q[metric] - v).abs().argsort()[:n]]
        for ax, (_, r) in zip(row, pick.iterrows()):
            ax.imshow(cv2.imread(f"{DATASETS[r['dataset']]}/images/{r['name']}.png")[..., ::-1])
            ax.set_title(f'{metric[:8]}={r[metric]:.2f}', fontsize=8); ax.axis('off')
    plt.tight_layout(); plt.show()

show('occluded_fraction', [0.05, 0.1, 0.15, 0.25, 0.4])
show('glasses_fraction', [0.0, 0.005, 0.01, 0.05])
show('sharpness', [10, 25, 40, 80, 200])
show('black_fraction', [0.1, 0.2, 0.3, 0.4])

In [ ]:
THRESHOLDS = '--min_face_size 80 --max_black_fraction 0.8 --max_landmarks_outside 0.05 --min_sharpness 30 --max_occluded_fraction 0.12 --max_glasses_fraction 0.01 --max_hat_fraction 0.02'

dataset_args = ' '.join(f'--dataset {n}={f}/meta.csv,{OUT}/{n}_landmarks.npz,{OUT}/{n}_quality.csv' for n, f in DATASETS.items())
!python /kaggle/working/filter_and_split.py {dataset_args} --out_dir {OUT} {THRESHOLDS}

clean = pd.read_csv(f'{OUT}/clean_meta.csv')
print(pd.crosstab([clean['dataset'], clean['expression']], clean['split'], margins=True))

In [ ]:
rejected = pd.read_csv(f'{OUT}/rejected.csv', dtype={'name': str})
reasons = rejected['reason'].value_counts().index.tolist()
fig, axes = plt.subplots(len(reasons), 8, figsize=(16, 2.2 * len(reasons)), squeeze=False)
for row, reason in zip(axes, reasons):
    pick = rejected[rejected['reason'] == reason].sample(min(8, (rejected['reason'] == reason).sum()), random_state=0)
    for ax in row: ax.axis('off')
    for ax, (_, r) in zip(row, pick.iterrows()):
        ax.imshow(cv2.imread(f"{DATASETS[r['dataset']]}/images/{r['name']}.png")[..., ::-1]); ax.set_title(reason, fontsize=8)
plt.tight_layout(); plt.show()